In [1]:
# %pip menginstal ke environment KERNEL yang aktif — kompatibel di Colab, Jupyter, & VS Code.
%pip install trafilatura feedparser googlenewsdecoder aiohttp supabase python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import time
import random
import asyncio
import urllib.parse

import aiohttp
import feedparser
import trafilatura
from supabase import create_client
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from googlenewsdecoder import gnewsdecoder

In [3]:
PORTAL_TIER_1 = [
    "kompas.com",
    "detik.com",
    "tempo.co",
    "cnnindonesia.com",
    "cnbcindonesia.com",
    "antaranews.com",
    "liputan6.com",
    "republika.co.id",
    "suara.com",
    "tirto.id"
]

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Mobile/15E148 Safari/604.1"
]

In [4]:
# Panjang minimal teks artikel (char) agar dianggap valid -> buang fragmen nav/menu.
MIN_PANJANG_ISI = 120

# ====== Setelan anti-blokir Google News (atur di sini) ======
KONKURENSI = 1                 # decode SERIAL — kunci utama hindari 429 (JANGAN paralel)
JEDA_DECODE = 2                # detik jeda tiap decode (parameter bawaan gnewsdecoder)
MAKS_DECODE_PER_BATCH = 60     # decode maksimal N kandidat TERBARU per siklus
# ============================================================

# Cache lintas-batch: link Google News yang SUDAH berhasil di-decode -> tidak
# diulang. RSS sering memunculkan berita yang sama tiap 15 menit, jadi ini
# memfokuskan kuota decode pada berita yang benar-benar baru.
_sudah_didecode = set()
# Status berbagi antar-coroutine (dict mutable agar tak perlu 'global').
_state = {"diblokir": False}


async def proses_artikel_async(session, item, sem, stat):
    """Per-artikel: decode URL Google News -> cek URL sampah -> fetch -> ekstrak teks.

    Mengembalikan `item` terisi (dict) bila sukses, atau None bila gagal.
    """
    async with sem:
        # Bila batch ini sudah kena blokir (429/sorry), STOP decode sisanya agar
        # tidak memperparah. Akan dicoba lagi pada siklus berikutnya.
        if _state["diblokir"]:
            stat["dilewati_blokir"] += 1
            return None

        await asyncio.sleep(random.uniform(0.5, 1.5))
        # 1) Decode redirect Google News -> URL portal asli. interval= memberi jeda
        #    di dalam library agar laju ke endpoint decode Google tetap sopan.
        try:
            decode = await asyncio.to_thread(gnewsdecoder, item["google_link"], interval=JEDA_DECODE)
            if decode.get("status"):
                url_asli = decode.get("decoded_url")
            else:
                url_asli = None
                pesan = str(decode.get("message", "")).lower()
                if "429" in pesan or "sorry" in pesan or "too many" in pesan:
                    _state["diblokir"] = True   # backoff: hentikan sisa batch
        except Exception:
            url_asli = None
        if not url_asli:
            stat["gagal_decode"] += 1
            return None

        # Tandai sudah berhasil di-decode -> tidak diulang di batch berikutnya.
        _sudah_didecode.add(item["google_link"])
        if len(_sudah_didecode) > 20000:        # cegah membengkak tak terbatas
            _sudah_didecode.clear()
        item["URL"] = url_asli

        # 2) Buang URL sampah (index/tag) yang lolos filter judul.
        if any(p in url_asli.lower() for p in pola_url_sampah):
            stat["sampah_url"] += 1
            return None

        # 3) Fetch + ekstrak teks artikel.
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
            "Referer": "https://www.google.com/",
        }
        try:
            async with session.get(url_asli, headers=headers,
                                   timeout=aiohttp.ClientTimeout(total=15)) as response:
                if response.status != 200:
                    stat["gagal_http"] += 1
                    return None
                html = await response.text()
                # trafilatura.extract bersifat blocking -> jalankan di thread terpisah.
                teks = await asyncio.to_thread(trafilatura.extract, html)
        except Exception:
            stat["gagal_fetch"] += 1
            return None

        if not teks or len(teks) < MIN_PANJANG_ISI:
            stat["terlalu_pendek"] += 1
            return None

        item["Isi_Berita"] = teks
        return item

In [5]:
def hapus_watermark_portal(judul_bawaan_google, nama_portal):
    """Menghapus watermark ' - Nama Portal' yang ditambahkan Google News pada judul."""
    suffix_google = f" - {nama_portal}"
    if judul_bawaan_google.endswith(suffix_google):
        return judul_bawaan_google[:-len(suffix_google)].strip()
    # Fallback: buang potongan ' - ...' terakhir bila nama portal tidak sama persis.
    return re.sub(r'\s*-\s*[^-]+$', '', judul_bawaan_google).strip()

In [6]:
pola_mutlak = [
    # Detik & Kompas
    "berita dan informasi",
    "kabar akurat terpercaya",
    "timeline berita terbaru",
    "berita harian",

    # Liputan6, Suara, Tirto
    "kumpulan berita",
    "kumpulan artikel",
    "top 3 berita",

    # General Index Pages (Semua Portal)
    "indeks berita",
    "berita terpopuler",
    "top news",

    # Non-Berita / Tabular Data (Bikin AI bingung)
    "jadwal sholat",
    "prakiraan cuaca",
    "kurs valas"
]

pola_awalan = [
    # Awalan SEO Umum (Detik, Kompas, Republika)
    "berita terkini",
    "berita terbaru",
    "terkini dan terbaru",

    # Awalan Khas Liputan6, Tempo, Suara
    "berita hari ini",
    "kabar terbaru",
    "headline hari ini",

    # Awalan Khas CNN & CNBC
    "fokus berita",
    "kabar harian"
]

kata_haram = ["zodiak", "ramalan", "promo", "diskon", "lirik lagu"]

pola_url_sampah = ["/tag/", "/tags/", "/indeks/", "/index/"]

In [7]:
def apakah_berita_sampah(judul, url):
    """True bila judul/url terindikasi bukan berita substantif (index, tag, SEO filler)."""
    if not judul or not judul.strip():
        return True
    if len(judul.strip().split()) < 3:
        return True
    if any(p in url for p in pola_url_sampah):
        return True
    if any(kata in judul for kata in kata_haram):
        return True
    if any(pola in judul for pola in pola_mutlak):
        return True
    # Pengecekan presisi tinggi: hanya di awal judul.
    if any(judul.startswith(pola) for pola in pola_awalan):
        return True
    return False

In [8]:
# ==========================================
# WORKER 1: FETCHING BERITA
# ==========================================
async def _ambil_feed_rss(portal):
    """Ambil 1 feed RSS Google News untuk satu portal (feedparser blocking -> thread)."""
    query_encoded = urllib.parse.quote_plus(f"site:{portal} when:1h")
    url_rss = f"https://news.google.com/rss/search?q={query_encoded}&hl=id&gl=ID&ceid=ID:id"
    return await asyncio.to_thread(feedparser.parse, url_rss)


async def agregator_seluruh_berita_tier1_async():
    """Kumpulkan URL dari RSS Google News lalu decode + ekstrak teks (serial, anti-blokir)."""
    print(f"\n{'='*40}")
    print(f"🔄 BATCH: {datetime.now(ZoneInfo('Asia/Jakarta')).strftime('%H:%M:%S WIB')} | ASYNC SCRAPER")
    print(f"{'='*40}")

    stat = {"sampah_judul": 0, "gagal_decode": 0, "sampah_url": 0,
            "gagal_http": 0, "gagal_fetch": 0, "terlalu_pendek": 0, "dilewati_blokir": 0}

    # Reset status blokir untuk batch ini (kalau sebelumnya sempat 429, dicoba lagi).
    _state["diblokir"] = False

    # FASE 1: Ambil SEMUA feed RSS paralel (ini ke RSS, bukan endpoint decode -> aman),
    #         lalu saring sampah berbasis JUDUL (tanpa jaringan).
    feeds = await asyncio.gather(*[_ambil_feed_rss(p) for p in PORTAL_TIER_1])

    items, link_terlihat = [], set()
    for feed in feeds:
        for entry in feed.entries:
            try:
                link = entry.link
                if link in link_terlihat:          # buang duplikat dalam-batch sedini mungkin
                    continue
                link_terlihat.add(link)

                nama_portal = entry.source.title if "source" in entry else "Tidak diketahui"
                judul = hapus_watermark_portal(entry.title, nama_portal)

                # Filter judul dulu (murah) -> hemat panggilan decode yang mahal.
                if apakah_berita_sampah(judul.lower(), ""):
                    stat["sampah_judul"] += 1
                    continue

                # published_parsed (UTC) bisa None -> fallback ke waktu sekarang (WIB).
                if entry.get("published_parsed"):
                    waktu = (datetime(*entry.published_parsed[:6]) + timedelta(hours=7)) \
                        .strftime("%Y-%m-%d %H:%M:%S")
                else:
                    waktu = datetime.now(ZoneInfo("Asia/Jakarta")).strftime("%Y-%m-%d %H:%M:%S")

                items.append({"Judul": judul, "Portal": nama_portal, "Waktu_Rilis": waktu,
                              "google_link": link, "URL": "", "Isi_Berita": ""})
            except Exception:
                continue

    if not items:
        print("INFO: Tidak ada berita baru.")
        return []

    # FASE 1.5: Prioritaskan TERBARU -> lewati yang sudah pernah di-decode -> batasi jumlah.
    #           Inilah inti "realtime tanpa terblokir": decode sedikit tapi yang baru,
    #           serial + berjeda, setiap 15 menit.
    items.sort(key=lambda x: x["Waktu_Rilis"], reverse=True)
    items_baru = [it for it in items if it["google_link"] not in _sudah_didecode]
    dilewati_cache = len(items) - len(items_baru)
    kandidat = items_baru[:MAKS_DECODE_PER_BATCH]

    # FASE 2: Decode + fetch SERIAL (konkurensi rendah) supaya tidak memicu 429.
    sem = asyncio.Semaphore(KONKURENSI)
    async with aiohttp.ClientSession() as session:
        hasil = await asyncio.gather(
            *[proses_artikel_async(session, it, sem, stat) for it in kandidat]
        )

    # FASE 3: Kumpulkan yang valid + dedup berdasarkan URL asli (cegah duplikat upsert).
    semua_berita_valid, url_terlihat = [], set()
    for r in hasil:
        if r and r["URL"] not in url_terlihat:
            url_terlihat.add(r["URL"])
            semua_berita_valid.append(r)

    semua_berita_valid.sort(key=lambda x: x["Waktu_Rilis"], reverse=True)

    print("--- HASIL PENARIKAN ASINKRON ---")
    print(f"Judul-lolos: {len(items)} | sudah-pernah-decode (skip): {dilewati_cache} "
          f"| diproses batch ini: {len(kandidat)}  ->  Valid: {len(semua_berita_valid)}")
    print(f"  Dibuang: url-sampah={stat['sampah_url']} decode-gagal={stat['gagal_decode']} "
          f"http!=200={stat['gagal_http']} fetch-gagal={stat['gagal_fetch']} "
          f"terlalu-pendek={stat['terlalu_pendek']} dilewati-blokir={stat['dilewati_blokir']}")
    if _state["diblokir"]:
        print("  ⚠️ Google membatasi (429). Sebagian dilewati; otomatis dicoba lagi siklus berikutnya.")
    print(f"✅ SUKSES: {len(semua_berita_valid)} berita valid.")

    return semua_berita_valid

In [9]:
import os
from dotenv import load_dotenv

# Baca kredensial dari file .env (bukan hardcode). Scraper MENULIS ke DB (upsert
# ke tabel_berita) yang kini dilindungi RLS. Pakai SERVICE_ROLE agar operasi
# tulis bypass RLS — anon akan ditolak (error 42501 violates RLS policy).
load_dotenv()
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_KEY"]

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [10]:
async def simpan_berita_ke_db_async(array_berita_bersih):
    """
    Menyimpan berita ke Supabase secara batch & asinkron.
    Secara otomatis mengabaikan URL duplikat.
    """
    if not array_berita_bersih:
        return

    print(f"\n[INGESTION] Menyuntikkan {len(array_berita_bersih)} berita ke Supabase secara masal...")

    data_untuk_dimasukkan = [
        {
            "judul": b["Judul"],
            "isi_teks": b["Isi_Berita"],
            "portal_sumber": b["Portal"],
            "url_asli": b["URL"],
            "waktu_rilis": b["Waktu_Rilis"],
            "status_proses": 0
        }
        for b in array_berita_bersih
    ]

    # Wrapper fungsi sinkron Supabase agar tidak memblokir event loop
    def jalankan_upsert():
        return supabase.table("tabel_berita").upsert(
            data_untuk_dimasukkan,
            on_conflict="url_asli",
            ignore_duplicates=True # Abaikan baris baru jika URL sudah ada di DB
        ).execute()

    try:
        # Eksekusi upsert di thread terpisah
        hasil = await asyncio.to_thread(jalankan_upsert)

        berita_masuk = len(hasil.data)
        print(f"[INGESTION] Sukses! {berita_masuk} berita (baru) masuk ke Supabase.")
    except Exception as e:
        print(f"⚠️ Gagal melakukan Batch Upsert: {e}")

In [11]:
async def run_worker_1_periodically():
    interval_detik = 900  # 900 dtk = 15 menit antar siklus (near-realtime, ramah rate-limit)

    while True:
        waktu_mulai = time.time()

        # 1. Eksekusi scraper.
        data_berita = await agregator_seluruh_berita_tier1_async()

        for i, b in enumerate(data_berita[:5], start=1):
            print(f"{i:02d}. {b['Judul']} [{b['Waktu_Rilis']}]")

        # 2. Simpan ke Supabase secara asinkron.
        if data_berita:
            await simpan_berita_ke_db_async(data_berita)

        # 3. Hitung durasi dan sisa waktu tidur.
        durasi_eksekusi = time.time() - waktu_mulai
        waktu_tidur = interval_detik - durasi_eksekusi

        if waktu_tidur > 0:
            print(f"\n[✅ Siklus Selesai ({durasi_eksekusi:.2f} detik). Tidur selama {waktu_tidur:.2f} detik...]")
            await asyncio.sleep(waktu_tidur)
        else:
            print(f"\n[⚠️ Bahaya: Eksekusi melampaui interval 15 menit ({durasi_eksekusi:.2f} detik). Langsung gas siklus baru!]")

In [ ]:
if __name__ == "__main__":
    # Karena Jupyter sudah memiliki event loop yang berjalan, panggil fungsi asinkron dengan await
    await run_worker_1_periodically()


🔄 BATCH: 19:16:18 WIB | ASYNC SCRAPER
--- HASIL PENARIKAN ASINKRON ---
Judul-lolos: 373 | sudah-pernah-decode (skip): 0 | diproses batch ini: 60  ->  Valid: 55
  Dibuang: url-sampah=0 decode-gagal=1 http!=200=3 fetch-gagal=0 terlalu-pendek=1 dilewati-blokir=0
✅ SUKSES: 55 berita valid.
01. Italia Tak Terima Disebut Sekjen NATO Bantu AS Serang Iran [2026-06-26 19:14:27]
02. Lima Rekomendasi SCI agar Rantai Pasok Tetap Tangguh Hadapi Disrupsi Global [2026-06-26 19:13:25]
03. Jepang Siapkan Ibu Kota Kedua Demi Dukung Tokyo [2026-06-26 19:13:15]
04. Fakta Kebakaran Hebat di Wonosobo: 8 Rumah Hangus dan 28 Warga Kehilangan Tempat Tinggal [2026-06-26 19:13:00]
05. Arda Guler: Turki Bisa Pulang dengan Bangga Usai Bungkam Tuan Rumah AS [2026-06-26 19:12:15]

[INGESTION] Menyuntikkan 55 berita ke Supabase secara masal...
[INGESTION] Sukses! 51 berita (baru) masuk ke Supabase.

[✅ Siklus Selesai (388.85 detik). Tidur selama 511.15 detik...]
